# Business Insights

This notebook generates business insights and visualizations from the analysis.


In [ ]:
import sys
import os
sys.path.append(os.path.join(os.path.dirname(os.getcwd()), 'src'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from utils import load_data
from eda_plots import plot_feature_importance, plot_model_comparison

# Load data
features_path = '../data/processed/grocery_features.csv'
discount_path = '../data/processed/discount_calendar.csv'

df_features = load_data(features_path)
df_discount = load_data(discount_path)


In [ ]:
# Key insights summary
print("="*60)
print("BUSINESS INSIGHTS SUMMARY")
print("="*60)

if df_discount is not None:
    # Check if it's Season × Product calendar or time-series calendar
    if 'Season' in df_discount.columns and 'Product ID' in df_discount.columns:
        disc_col = 'Optimal_Discount' if 'Optimal_Discount' in df_discount.columns else 'optimal_discount'
        rev_col = 'Estimated_Revenue' if 'Estimated_Revenue' in df_discount.columns else 'predicted_revenue'
        pred_col = 'Predicted_Demand' if 'Predicted_Demand' in df_discount.columns else 'predicted_sales'
        
        print(f"\n1. Total Product × Season Combinations: {len(df_discount)}")
        print(f"2. Average Optimal Discount: {df_discount[disc_col].mean():.2f}%")
        print(f"3. Discount Range: {df_discount[disc_col].min():.0f}% - {df_discount[disc_col].max():.0f}%")
        print(f"4. Total Estimated Revenue: ${df_discount[rev_col].sum():,.2f}")
        print(f"5. Average Predicted Demand per combination: {df_discount[pred_col].mean():.2f}")
        
        # Show top recommendations by season
        print("\n6. Top Discount Recommendations by Season:")
        for season in df_discount['Season'].unique():
            season_data = df_discount[df_discount['Season'] == season]
            avg_disc = season_data[disc_col].mean()
            print(f"   - {season}: Average discount {avg_disc:.1f}% ({len(season_data)} products)")
    else:
        disc_col = 'optimal_discount' if 'optimal_discount' in df_discount.columns else 'Optimal_Discount'
        rev_col = 'predicted_revenue' if 'predicted_revenue' in df_discount.columns else 'Estimated_Revenue'
        pred_col = 'predicted_sales_with_discount' if 'predicted_sales_with_discount' in df_discount.columns else 'Predicted_Demand'
        
        print(f"\n1. Average Optimal Discount: {df_discount[disc_col].mean():.2f}%")
        print(f"2. Total Predicted Revenue: ${df_discount[rev_col].sum():,.2f}")
        print(f"3. Average Daily Sales (with discount): {df_discount[pred_col].mean():.2f}")

print("\n" + "="*60)
print("KEY RECOMMENDATIONS")
print("="*60)
print("1. Implement dynamic pricing based on demand forecasts")
print("2. Use Season × Product discount calendar for optimal pricing strategy")
print("3. Optimize discount timing for maximum revenue")
print("4. Monitor seasonal patterns for inventory planning")
print("5. Bundle low-demand products with high-demand products")
print("6. Adjust discount strategy during epidemic periods")


In [ ]:
# Visualize discount calendar results
if df_discount is not None:
    # If it's a Season × Product calendar, create a pivot table visualization
    if 'Season' in df_discount.columns and 'Product ID' in df_discount.columns:
        import matplotlib.pyplot as plt
        import seaborn as sns
        
        # Create pivot table for heatmap
        pivot = df_discount.pivot_table(
            values='Optimal_Discount' if 'Optimal_Discount' in df_discount.columns else 'optimal_discount',
            index='Product ID',
            columns='Season',
            aggfunc='mean'
        )
        
        # Create heatmap
        fig, ax = plt.subplots(figsize=(12, max(8, len(pivot) * 0.5)))
        sns.heatmap(pivot, annot=True, fmt='.0f', cmap='YlOrRd', 
                    cbar_kws={'label': 'Discount (%)'}, ax=ax)
        ax.set_title('Product × Seasonal Discount Calendar', 
                     fontsize=16, fontweight='bold')
        ax.set_xlabel('Season', fontsize=12)
        ax.set_ylabel('Product ID', fontsize=12)
        plt.tight_layout()
        plt.savefig('../reports/figures/discount_calendar.png', dpi=300, bbox_inches='tight')
        plt.show()
        print("Discount calendar heatmap saved to ../reports/figures/discount_calendar.png")
    else:
        # For time-series discount calendar
        fig, axes = plt.subplots(2, 1, figsize=(14, 10))
        
        date_col = 'date' if 'date' in df_discount.columns else df_discount.columns[0]
        disc_col = 'optimal_discount' if 'optimal_discount' in df_discount.columns else 'Optimal_Discount'
        rev_col = 'predicted_revenue' if 'predicted_revenue' in df_discount.columns else 'Estimated_Revenue'
        
        # Plot optimal discounts over time
        axes[0].plot(df_discount[date_col], df_discount[disc_col], linewidth=2)
        axes[0].set_title('Optimal Discount Over Time', fontsize=14, fontweight='bold')
        axes[0].set_xlabel('Date')
        axes[0].set_ylabel('Discount (%)')
        axes[0].grid(True, alpha=0.3)
        
        # Plot predicted revenue
        axes[1].plot(df_discount[date_col], df_discount[rev_col], linewidth=2, color='green')
        axes[1].set_title('Predicted Revenue Over Time', fontsize=14, fontweight='bold')
        axes[1].set_xlabel('Date')
        axes[1].set_ylabel('Revenue ($)')
        axes[1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.savefig('../reports/figures/discount_calendar.png', dpi=300, bbox_inches='tight')
        plt.show()
        print("Discount calendar plot saved to ../reports/figures/discount_calendar.png")
